In [0]:
%python
# --- 
# PROJECT: NYC Ride Analytics
# STAGE: Silver Layer (Data Cleaning & Enrichment)
# OBJECTIVE: Transform raw taxi data into high-quality, query-ready data
# ---

# --- SECURITY CONFIGURATION (REDACTED FOR GITHUB) ---
# In a production environment, these are managed via Databricks Secret Scopes
# Example: client_secret = dbutils.secrets.get(scope="analytics-scope", key="adls-secret")
client_id = "REDACTED_CLIENT_ID"
tenant_id = "REDACTED_TENANT_ID"
client_secret = "REDACTED_CLIENT_SECRET"
storage_account_name = "datalakerides" 

# Setting session-level configurations for OAuth2 authentication with Azure AD (Entra ID)
# This allows Spark to communicate directly with ADLS Gen2 without mounting
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

from pyspark.sql.functions import col, unix_timestamp, round

# Define the source path (Bronze) from the 'raw-data' container
bronze_path = f"abfss://raw-data@{storage_account_name}.dfs.core.windows.net/ride_data_raw"

# Load the Delta table from the Bronze layer
df_bronze = spark.read.format("delta").load(bronze_path)

print("Status: Successfully loaded records from Bronze (Credentials Redacted).")

Status: Successfully loaded records from Bronze.


In [0]:
%python
# ---
# AUDIT: Checking the current schema of our Bronze data 
# This ensures our transformation logic matches the source column names exactly.
# ---

df_bronze.printSchema()

# We also show the first 5 rows to see the actual data values
display(df_bronze.limit(5))

root
 |-- vendor_id: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- rate_code_id: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)



vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,rate_code_id,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,total_amount
VTS,2009-11-07T19:44:00Z,2009-11-07T19:49:00Z,2,0.74,-73.992127,40.734658,null,null,-73.99197,40.729115,CASH,4.5,0.0,0.5,0.0,0.0,5.0
VTS,2009-11-08T02:06:00Z,2009-11-08T02:12:00Z,1,1.04,-74.008553,40.719682,null,null,-74.01615,40.709963,CASH,5.7,0.5,0.5,0.0,0.0,6.7
VTS,2009-11-08T03:57:00Z,2009-11-08T04:09:00Z,1,4.05,-74.007742,40.717097,null,null,-73.986758,40.768467,CASH,11.7,0.5,0.5,0.0,0.0,12.7
VTS,2009-11-05T12:56:00Z,2009-11-05T12:58:00Z,1,0.33,-73.992995,40.71258,null,null,-73.992995,40.71258,CASH,3.3,0.0,0.5,0.0,0.0,3.8
VTS,2009-11-05T08:35:00Z,2009-11-05T08:41:00Z,1,0.6,-74.00921,40.712933,null,null,-74.002663,40.716882,CASH,4.5,0.0,0.5,0.0,0.0,5.0


In [0]:
%python
# ---
# TRANSFORMATION LOGIC:
# Using the audited schema to ensure 100% accuracy in our calculations.
# 1. CLEANING: Removing invalid records (0 distance or 0 passengers).
# 2. FEATURE ENGINEERING: Calculating trip duration in minutes for the Gold layer.
# ---

# Step 1: Filter out the 'noise' (non-ride pings or errors)
df_filtered = df_bronze.filter(
    (col("passenger_count") > 0) & 
    (col("trip_distance") > 0)
)

# Step 2: Calculate duration. Since our schema has 'timestamp' types, 
# Spark can subtract them directly after converting to unix_timestamp.
df_enriched = df_filtered.withColumn(
    "trip_duration_minutes", 
    round((unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))) / 60, 2)
)

# Step 3: Selecting only the core columns needed for our business dashboard
df_silver_final = df_enriched.select(
    "vendor_id",
    "pickup_datetime",
    "dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "trip_duration_minutes"
)

# Show the results to confirm the duration calculation worked
display(df_silver_final.limit(10))

vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,trip_duration_minutes
VTS,2009-11-07T19:44:00Z,2009-11-07T19:49:00Z,2,0.74,4.5,0.0,5.0,5.0
VTS,2009-11-08T02:06:00Z,2009-11-08T02:12:00Z,1,1.04,5.7,0.0,6.7,6.0
VTS,2009-11-08T03:57:00Z,2009-11-08T04:09:00Z,1,4.05,11.7,0.0,12.7,12.0
VTS,2009-11-05T12:56:00Z,2009-11-05T12:58:00Z,1,0.33,3.3,0.0,3.8,2.0
VTS,2009-11-05T08:35:00Z,2009-11-05T08:41:00Z,1,0.6,4.5,0.0,5.0,6.0
VTS,2009-11-05T08:15:00Z,2009-11-05T08:24:00Z,2,1.37,6.5,4.0,11.0,9.0
VTS,2009-11-08T13:48:00Z,2009-11-08T14:02:00Z,1,3.03,10.5,0.0,11.0,14.0
VTS,2009-11-08T19:26:00Z,2009-11-08T19:38:00Z,1,2.32,8.5,0.0,9.0,12.0
VTS,2009-11-05T22:25:00Z,2009-11-05T22:40:00Z,1,4.35,13.7,0.0,14.7,15.0
VTS,2009-11-05T19:10:00Z,2009-11-05T19:18:00Z,1,1.63,6.9,0.0,8.4,8.0


In [0]:
%python
# ---
# SINK: Persisting the high-quality Silver data to Azure Data Lake
# ---

silver_path = f"abfss://transformed-data@{storage_account_name}.dfs.core.windows.net/ride_data_silver"

# Overwrite the Silver layer with the final cleaned dataframe
df_silver_final.write.format("delta").mode("overwrite").save(silver_path)

print(f"SUCCESS: Silver table is now live in 'transformed-data' container.")

SUCCESS: Silver table is now live in 'transformed-data' container.
